## American Wars Kill Children

Data visualization for the first 20 days of American wars in the last 20 years

In [7]:
#importing the right stuff

import pandas as pd
import altair as alt

In [8]:
#read the data

df = pd.read_csv("america-kills-children.csv")
df.head(20)

,conflict_name,operation_name,conflict_start_date,20_day_period,time_period,date_recorded,children_killed_min_first_20_days,children_killed_max_first_20_days,perpetrators,source
0,Afghanistan (2001),Operation Enduring Freedom (phase 1)\nOperatio...,"October 7, 2001","October 7, 2001 to\nOctober 27, 2001",2001 - 2021,"October 26, 2003",67.0,67.0,U.S.,https://dcas.dmdc.osd.mil/dcas/app/conflictCas...
1,Iraq War / Second Gulf War \n(2003),Opreation Iraqi Freedom (CENTCOM)\nOperation T...,"March 20, 2003","March 20, 2003 to\nApril 9, 2003",2003 - 2010,"April 9, 2003",78.0,86.0,U.S. \nU.K.\nAustralia,https://www.centcom.mil/MEDIA/NEWS-ARTICLES/Ne...
2,Pakistan Drone Strikes (2004),CIA's covert drone strikes program in FATA,"June 18, 2004","June 18, 2004 to\nJuly 8. 2004",2004 - 2018,"July 8, 2004",0.0,0.0,U.S.\nISI,https://www.thebureauinvestigates.com/projects...
3,Somalia (2007),Airstrikes against Al-Qaeda,"January 7, 2007","January 7, 2007 to\nJanuary 27, 2007",2007 - present,NaN,1.0,1.0,U.S.,https://airwars.org/conflict/us-forces-in-soma...
4,Libya (2011),Operation Odyssey Dawn (US/AFRICOM)\nOperation...,"March 17, 2011","March 17, 2011 to\nApril 6, 2011",2011,"February, 2012",0.0,0.0,NATO,https://www.nato.int/en/what-we-do/operations-...
5,Syria (2014),Operation Inherent Resolve (CENTCOM),"October 17, 2014","October 17, 2014 to\nNovember 6, 2014",2014 - 2024,"October 23, 2014",8.0,8.0,U.S.\nSaudi Arabia\nU.A.E.\nJordan\nBahrain\nQ...,https://www.inherentresolve.mil/WHO-WE-ARE/His...
6,KSA/UAE-Yemen (2015),Operation Decisive Storm (KSA)\nOperation Rest...,"March 26, 2015","March 26, 2015 to\nApril 15, 2015",March - April 2015,NaN,NaN,NaN,Saudi Arabia\nU.A.E.\nU.S.,https://edition.cnn.com/interactive/2018/09/wo...
7,Ukraine (2022),Operation Atlantic Resolve (started in 2014),"February 24, 2022","February 24, 2022 to \nMarch 16, 2022",2014-2022,NaN,NaN,NaN,Russia\nU.S.,NaN
8,Israel-Iran (The 12 Day war)\n(2025),Operation Rising Lion (IDF),"June 13, 2025","June 13, 2025 to\nJune 24, 2025","June 13 - 24, 2025","June 24, 2025",13.0,13.0,Israel\nU.S.,https://shafaq.com/en/Middle-East/Over-600-civ...
9,US-Yemen (2025),Airstrikes against Houthis,"March 15, 2025","March 15, 2025 to \nApril 05, 2025",NaN,"March 28, 2025",5.0,9.0,U.S.,https://www.ap.org/news-highlights/spotlights/...


In [9]:
#keeping only the max figures bec that's what we want to visualize

df = df[df["children_killed_max_first_20_days"].notna()].copy()

#dropping the NaN values for now

df = df[df["children_killed_max_first_20_days"] > 0].dropna(subset=["children_killed_max_first_20_days"]).copy()
#making sure the number of deaths are always seen as an integer

df["children"] = df["children_killed_max_first_20_days"].astype(int)


In [10]:
#for the chart, fixing how many children appear in each row before moving on to the next line

COLS = 20

rows = []
for _, r in df.iterrows():
    for i in range(r["children"]):
        rows.append({
            "conflict": r["conflict_name"],
            "children": r["children"],
            "col": i % COLS,
            "row": i // COLS,
        })

figures = pd.DataFrame(rows)
figures.head(10)

,conflict,children,col,row
0,Afghanistan (2001),67,0,0
1,Afghanistan (2001),67,1,0
2,Afghanistan (2001),67,2,0
3,Afghanistan (2001),67,3,0
4,Afghanistan (2001),67,4,0
5,Afghanistan (2001),67,5,0
6,Afghanistan (2001),67,6,0
7,Afghanistan (2001),67,7,0
8,Afghanistan (2001),67,8,0
9,Afghanistan (2001),67,9,0


In [11]:
#drawing the chart, figuring every row of the viz should have 20 

#we want our dots to look like children so we save the child_shape

child_shape = "M0,-10 C4,-10 6,-7 6,-4 C6,-1 4,0 0,0 C-4,0 -6,-1 -6,-4 C-6,-7 -4,-10 0,-10Z M-5,0 L-5,12 L5,12 L5,0Z M-8,1 L-5,1 L-5,9 L-8,9Z M5,1 L8,1 L8,9 L5,9Z M-4,12 L-4,20 L-1,20 L-1,12Z M1,12 L1,20 L4,20 L4,12Z" 


In [20]:
#fixing the rows per line

COL = 20

#how tall each panel should be

ROW_HEIGHT = 20

# calculate how many rows each conflict needs

rows_per_conflict = figures.groupby("conflict")["row"].max() + 1

#got to make sure iran-israel panel is the tallest
iran_rows = figures[figures["conflict"].str.contains("2026")]["row"].max() + 1
panel_height = int(iran_rows * ROW_HEIGHT)


In [21]:

#now to chart the chart

chart = alt.Chart(figures).mark_point(
    shape=child_shape,
    filled=True,
    size=2,
    opacity=1.0
).encode(
    x=alt.X("col:Q",
        axis=None,
        scale=alt.Scale(domain=[0, COLS])),
    y=alt.Y("row:Q",
        axis=None,
        scale=alt.Scale(domain=[0, max_rows])),
    color=alt.value("#C0392B"),
    facet=alt.Facet(
        "conflict:N",
        columns=1,
        header=alt.Header(
            labelFont="IBM Plex Mono",
            labelFontSize=11,
            labelColor="#1a1410",
            labelAlign="left",
            labelAnchor="start"
        )
    ),
    tooltip=[
        alt.Tooltip("conflict:N"),
        alt.Tooltip("children:Q"),
    ]
).properties(
    width=340,
    height=panel_height
)

chart

alt.Chart(...)